# Planning Analyses

The cuts the reconciliation doesn't surface — concentration, readiness gates, schedule slippage,
sqft-weighted risk, cross-system drift, and diff aging. Each is a small, standalone read off the
same shared loaders (`recon_lib.py`). SFDC-only cells run from the CSV; cross-system cells need a
token.

In [ ]:
import recon_lib as rl
import pandas as pd, datetime as dt, os, glob, json
pd.set_option("display.max_columns", None, "display.width", 200)
TODAY = dt.date.today()
sfdc = rl.load_sfdc("data/sfdc_report_2026-06-22.csv")
at   = rl.load_airtable()
print(f"SFDC rows: {len(sfdc)} | Airtable: {'loaded' if at is not None else 'no token'}")
total_sqft = sfdc["sqft_num"].sum()

## 1. Backlog concentration (Pareto) — whom to coordinate with first

By site-service count *and* by sqft, because they tell different stories: count drives coordination
load, sqft drives the 100M goal.

In [ ]:
def pareto(df, key, n=12):
    g = df.groupby(df[key].replace("", "(blank)")).agg(
        site_services=("site_service", "size"),
        sqft=("sqft_num", "sum")).sort_values("site_services", ascending=False)
    g["ss_%"]   = (g["site_services"] / len(df) * 100).round(1)
    g["sqft_%"] = (g["sqft"] / total_sqft * 100).round(1)
    return g.head(n)

print("TOP ACCOUNTS:")
acct = pareto(sfdc, "account")
top10 = acct.head(10)
print(f"  Top 10 accounts = {top10['ss_%'].sum():.0f}% of site services, {top10['sqft_%'].sum():.0f}% of sqft")
display(acct)
print("\nTOP JOB OWNERS:")
display(pareto(sfdc, "owner"))

## 2. Readiness-gate proxy v0 — leading indicator before the prereq owner map exists

We don't have the 15–20 prerequisites wired yet, but the fields we already pull are themselves
gates: a site with no owner / no target date / no status / no AT record is *not ready* and is a
surprise waiting to happen.

In [ ]:
gates = {
    "Missing Job Owner":      (sfdc["owner"].map(rl.norm_text) == "").sum(),
    "Missing Job Status":     (sfdc["job_status"].map(rl.norm_text) == "").sum(),
    "Missing Target Go-Live": sfdc["target_golive_d"].isna().sum(),
    "Zero / missing sqft":    (sfdc["sqft_num"] == 0).sum(),
}
if at is not None:
    keyed = set(at.dropna(subset=["job_id_key"])["job_id_key"])
    gates["No matching Airtable record"] = (~sfdc["job_id_key"].isin(keyed)).sum()
g = pd.DataFrame.from_dict(gates, orient="index", columns=["site_services"])
g["pct_of_backlog"] = (g["site_services"] / len(sfdc) * 100).round(1)
g.sort_values("site_services", ascending=False)

## 3. Schedule slippage — target vs. actual

For the backlog (pre-live), how far past target are the late ones; for live jobs (Airtable),
actual − target.

In [ ]:
late = sfdc[sfdc["target_golive_d"].notna() & (sfdc["target_golive_d"] < TODAY)].copy()
late["days_late"] = late["target_golive_d"].map(lambda d: (TODAY - d).days)
print(f"Backlog sites already past target: {len(late)}")
if len(late):
    print(late["days_late"].describe().round(0).to_string())
    print("\nMost overdue (sqft-weighted view):")
    display(late.sort_values("sqft_num", ascending=False)
                [["site_service","account","sqft_num","target_golive","days_late"]].head(10))

if at is not None:
    lv = at[at["actual_golive_d"].notna() & at["target_golive_d"].notna()].copy()
    lv["slip_days"] = [(a - t).days for a, t in zip(lv["actual_golive_d"], lv["target_golive_d"])]
    print(f"\nLive jobs with both dates: {len(lv)} | median slip {lv['slip_days'].median():.0f} days")

## 4. Sqft-weighted risk — rank the backlog by what moves the 100M goal

Every count above is unweighted. Weighting the *at-risk* backlog (late or scheduled past 9/30) by
sqft tells you where the goal-relevant exposure actually sits.

In [ ]:
risk = sfdc[(sfdc["target_golive_d"].isna()) |
            (sfdc["target_golive_d"] < TODAY) |
            (sfdc["target_golive_d"] > rl.GOAL_DATE)].copy()
print(f"At-risk site services: {len(risk)} ({len(risk)/len(sfdc)*100:.0f}%) | "
      f"{risk['sqft_num'].sum():,.0f} sqft ({risk['sqft_num'].sum()/total_sqft*100:.0f}% of backlog sqft)")
print("\nAt-risk sqft by account (top 12):")
display(risk.groupby("account").agg(sites=("site_service","size"), at_risk_sqft=("sqft_num","sum"))
            .sort_values("at_risk_sqft", ascending=False).head(12))

## 5. Cross-system drift by field *(requires token)*

Where the two systems disagree, *which attribute* drifts most — this names the war-room backlog.

In [ ]:
if at is None:
    print("No token — set AIRTABLE_API_TOKEN to compute cross-system drift.")
else:
    a = at.dropna(subset=["job_id_key"]).drop_duplicates("job_id_key").set_index("job_id_key")
    s = sfdc.dropna(subset=["job_id_key"]).drop_duplicates("job_id_key").set_index("job_id_key")
    shared = s.index.intersection(a.index)
    attrs = {"account":"Account","address":"Address","job_status":"Job Status",
             "job_type":"Job Type","target_golive":"Target Go-Live"}
    counts = {}
    for col, label in attrs.items():
        sv = s.loc[shared, col].map(rl.norm_text)
        av = a.loc[shared, col].map(rl.norm_text)
        counts[label] = int((sv != av).sum())
    drift = pd.DataFrame.from_dict(counts, orient="index", columns=["jobs_differing"])
    drift["pct_of_shared"] = (drift["jobs_differing"] / len(shared) * 100).round(1)
    print(f"Shared jobs: {len(shared)}")
    display(drift.sort_values("jobs_differing", ascending=False))

## 6. Diff aging — how long each diff has persisted

Reads the reconciliation notebook's `runs/diffs_*.json` snapshots. A diff present across many runs
is genuine drift; one that appears once is noise.

In [ ]:
snaps = sorted(glob.glob("runs/diffs_*.json"))
if not snaps:
    print("No runs/diffs_*.json yet — run reconcile_sfdc_airtable.ipynb a few times to accrue aging.")
else:
    from collections import Counter
    streak = Counter()
    for snap in snaps:
        for jid in json.load(open(snap)).get("diff_job_ids", []):
            streak[jid] += 1
    aging = (pd.Series(streak, name="runs_on_diff_list").sort_values(ascending=False)
               .to_frame())
    print(f"{len(snaps)} run snapshots | {len(aging)} jobs have differed at least once")
    print(f"Jobs differing in EVERY run (hard drift): {(aging['runs_on_diff_list']==len(snaps)).sum()}")
    display(aging.head(20))